In [1]:
import os
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import xarray as xr
import numpy as np

from tqdm import tqdm

In [2]:
# ==========================================
# 1. ПОДГОТОВКА ДАТАСЕТА
# ==========================================
class ERA5HighResDataset(Dataset):
    def __init__(self, nc_path):
        print("Загрузка 15 ГБ NetCDF файла... (потребуется от 32 ГБ ОЗУ)")
        ds = xr.open_dataset(nc_path)
        
        surf_vars = [
            '2m_temperature', 'mean_sea_level_pressure', '10m_u_component_of_wind',
            '10m_v_component_of_wind', 'total_precipitation_6hr', 'sea_surface_temperature',
            'total_column_water_vapour', 'total_cloud_cover'
        ]
        
        atm_vars = [
            'temperature', 'u_component_of_wind', 'v_component_of_wind',
            'geopotential', 'specific_humidity'
        ]
        
        print("Извлечение наземных переменных...")
        surf_arrays = [ds[v].values[:, np.newaxis, :, :] for v in surf_vars]
        
        print("Извлечение атмосферных переменных...")
        atm_arrays = [ds[v].values for v in atm_vars]
        
        # Закрываем исходный датасет для экономии памяти
        ds.close()
        del ds
        gc.collect()
        
        print("Склейка тензоров...")
        self.data = np.concatenate(surf_arrays + atm_arrays, axis=1)
        
        # Очищаем временные списки
        del surf_arrays
        del atm_arrays
        gc.collect()

        print("Заполнение NaN и нормализация...")
        self.data = np.nan_to_num(self.data, nan=0.0)
        
        self.mean = np.mean(self.data, axis=(0, 2, 3), keepdims=True)
        self.std = np.std(self.data, axis=(0, 2, 3), keepdims=True)
        self.std[self.std == 0] = 1.0 
        
        self.data = (self.data - self.mean) / self.std
        
        # Для AMP лучше всего подавать float32 (PyTorch сам приведет к fp16 на GPU)
        self.data = self.data.astype(np.float32)
        
        self.H_orig = self.data.shape[2]
        self.W_orig = self.data.shape[3]
        
        # Динамический паддинг (для 721x1440 высота станет 728)
        pad_H = (8 - self.H_orig % 8) % 8
        pad_W = (8 - self.W_orig % 8) % 8
        
        self.data = np.pad(self.data, ((0,0), (0,0), (0, pad_H), (0, pad_W)), mode='constant', constant_values=0)
        print(f"Форма данных для обучения: {self.data.shape}")

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return torch.tensor(self.data[idx])

In [7]:
class ConvVAE(nn.Module):
    def __init__(self, in_channels=28, latent_channels=128):
        super(ConvVAE, self).__init__()
        
        self.enc1 = nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1)
        self.enc2 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)
        self.enc3 = nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1)
        
        self.fc_mu = nn.Conv2d(256, latent_channels, kernel_size=3, padding=1)
        self.fc_var = nn.Conv2d(256, latent_channels, kernel_size=3, padding=1)
        
        self.dec_input = nn.Conv2d(latent_channels, 256, kernel_size=3, padding=1)
        
        self.dec1 = nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1)
        self.dec2 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        self.dec3 = nn.ConvTranspose2d(64, in_channels, kernel_size=4, stride=2, padding=1)
        
    def encode(self, x):
        h = F.leaky_relu(self.enc1(x))
        h = F.leaky_relu(self.enc2(h))
        h = F.leaky_relu(self.enc3(h))
        return self.fc_mu(h), self.fc_var(h)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def quantize(self, z):
        # Квантование до целых чисел с помощью Straight-Through Estimator (STE)
        # Это позволяет градиентам проходить через недифференцируемую функцию округления при backpropagation
        z_rounded = torch.round(z)
        return z + (z_rounded - z).detach()

    def decode(self, z):
        h = F.relu(self.dec_input(z))
        h = F.relu(self.dec1(h))
        h = F.relu(self.dec2(h))
        return self.dec3(h)

    def forward(self, x):
        mu, log_var = self.encode(x)
        # Оставляем reparameterize, если хотим сохранить стохастичность, 
        # либо можно просто использовать mu как z: z = mu
        z = self.reparameterize(mu, log_var) 
        z_quantized = self.quantize(z)
        recon_x = self.decode(z_quantized)
        
        # ВАЖНО: возвращаем квантованный и оригинальный z
        return recon_x, z_quantized, z

In [8]:
# # ==========================================
# # 3. ФУНКЦИЯ ПОТЕРЬ И ЦИКЛ ОБУЧЕНИЯ
# # ==========================================
# def vae_loss_function(recon_x, x, mu, log_var, H_orig, W_orig):
#     # Обрезаем черный паддинг
#     valid_recon = recon_x[:, :, :H_orig, :W_orig]
#     valid_x = x[:, :, :H_orig, :W_orig]
    
#     # MSE Loss - используем sum и делим на размер батча
#     recon_loss = F.mse_loss(valid_recon, valid_x, reduction='sum') / x.size(0)
    
#     # KLD Loss 
#     kld_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / x.size(0)
    
#     beta = 0.05  # Вес для KLD, для огромных картинок его нужно держать маленьким
#     return recon_loss + beta * kld_loss, recon_loss, kld_loss


In [9]:
import os
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
# Убедитесь, что импортированы ERA5HighResDataset и ConvVAE

def weather_physics_loss(recon_x, x, quantized_z, original_z, H_orig, W_orig, precipitation_idx=4):
    # 1. Отсекаем паддинг перед расчетом лоссов
    valid_recon = recon_x[:, :, :H_orig, :W_orig]
    valid_x = x[:, :, :H_orig, :W_orig]
    
    # 2. L_reconstruction (Базовый MSE)
    l_recon = F.mse_loss(valid_recon, valid_x, reduction='mean')
    
    # 3. L_gradient (Производные по осям X и Y)
    dx_true = torch.diff(valid_x, dim=3)
    dy_true = torch.diff(valid_x, dim=2)
    dx_recon = torch.diff(valid_recon, dim=3)
    dy_recon = torch.diff(valid_recon, dim=2)
    
    l_grad = F.mse_loss(dx_recon, dx_true) + F.mse_loss(dy_recon, dy_true)
    
    # 4. L_precipitation (Осадки находятся под индексом 4 в surface_vars)
    precip_recon = valid_recon[:, precipitation_idx, :, :]
    precip_true = valid_x[:, precipitation_idx, :, :]
    l_precip = F.mse_loss(precip_recon, precip_true)
    
    # 5. L_occupancy (Штраф за сильное отклонение при квантовании)
    l_occupancy = F.mse_loss(quantized_z, original_z.detach())
    
    # Итоговая сумма с физическими весами
    total_loss = l_recon + 0.015 * l_grad + 0.080 * l_precip + 0.010 * l_occupancy
    
    return total_loss, l_recon, l_grad, l_precip, l_occupancy


def main():
    data_path = 'data/era5_highres_sample_2019.nc'
    
    batch_size = 2 
    epochs = 50
    learning_rate = 1e-4
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Используемое устройство: {device}")
    if torch.cuda.is_available():
        print(f"Видеокарта: {torch.cuda.get_device_name(0)}")

    dataset = ERA5HighResDataset(data_path)
    dataloader = DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        drop_last=False,
        pin_memory=True, 
        num_workers=0
    )
    
    H_orig, W_orig = dataset.H_orig, dataset.W_orig

    model = ConvVAE(in_channels=28, latent_channels=128).to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    scaler = torch.amp.GradScaler('cuda')

    print("\nНачало обучения (Физический лосс + Mixed Precision)...")
    model.train()
    
    for epoch in range(epochs):
        # Аккумуляторы для новых лоссов
        epoch_total, epoch_recon, epoch_grad, epoch_precip, epoch_occup = 0, 0, 0, 0, 0
        
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1:03d}/{epochs}")
        
        for data in progress_bar:
            data = data.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            
            with torch.amp.autocast('cuda'):
                # Принимаем новые переменные из forward
                recon_batch, z_quantized, original_z = model(data)
                
                loss, l_recon, l_grad, l_precip, l_occup = weather_physics_loss(
                    recon_batch, data, z_quantized, original_z, H_orig, W_orig, precipitation_idx=4
                )
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            # Умножаем на batch_size, чтобы получить сумму лоссов
            bs = data.size(0)
            epoch_total += loss.item() * bs
            epoch_recon += l_recon.item() * bs
            epoch_grad += l_grad.item() * bs
            epoch_precip += l_precip.item() * bs
            epoch_occup += l_occup.item() * bs
            
            # Вывод компактной информации в tqdm
            progress_bar.set_postfix({
                'Total': f"{loss.item():.3f}",
                'Recon': f"{l_recon.item():.3f}",
                'Precip': f"{l_precip.item():.3f}"
            })
            
        num_samples = len(dataset) 
        
        # Рассчитываем средние значения лоссов за эпоху
        avg_total = epoch_total / num_samples
        avg_recon = epoch_recon / num_samples
        avg_grad = epoch_grad / num_samples
        avg_precip = epoch_precip / num_samples
        avg_occup = epoch_occup / num_samples
        
        # Вывод полной детализации в конце эпохи
        print(f"Epoch {epoch+1:03d}/{epochs} Summary:")
        print(f"  Total Loss : {avg_total:.5f}")
        print(f"  Recon (MSE): {avg_recon:.5f}")
        print(f"  Gradient   : {avg_grad:.5f}")
        print(f"  Precip     : {avg_precip:.5f}")
        print(f"  Occupancy  : {avg_occup:.5f}\n")

    print("Обучение завершено!")
    os.makedirs('models', exist_ok=True)
    torch.save(model.state_dict(), 'models/era5_highres_vae_weights.pth')
    print("Веса модели сохранены.")

if __name__ == "__main__":
    main()

Используемое устройство: cuda
Видеокарта: NVIDIA GeForce RTX 5070
Загрузка 15 ГБ NetCDF файла... (потребуется от 32 ГБ ОЗУ)
Извлечение наземных переменных...
Извлечение атмосферных переменных...
Склейка тензоров...
Заполнение NaN и нормализация...
Форма данных для обучения: (128, 28, 728, 1440)

Начало обучения (Физический лосс + Mixed Precision)...


Epoch 001/50: 100%|██████████| 64/64 [00:06<00:00, 10.36it/s, Total=0.728, Recon=0.643, Precip=1.024]


Epoch 001/50 Summary:
  Total Loss : 0.90221
  Recon (MSE): 0.81901
  Gradient   : 0.15048
  Precip     : 1.00144
  Occupancy  : 0.08339



Epoch 002/50: 100%|██████████| 64/64 [00:05<00:00, 11.85it/s, Total=0.592, Recon=0.506, Precip=1.048]


Epoch 002/50 Summary:
  Total Loss : 0.65955
  Recon (MSE): 0.58008
  Gradient   : 0.07759
  Precip     : 0.96850
  Occupancy  : 0.08342



...

Epoch 048/50 Summary:
  Total Loss : 0.05339
  Recon (MSE): 0.04342
  Gradient   : 0.02707
  Precip     : 0.10908
  Occupancy  : 0.08335



Epoch 049/50: 100%|██████████| 64/64 [00:05<00:00, 11.80it/s, Total=0.053, Recon=0.043, Precip=0.110]


Epoch 049/50 Summary:
  Total Loss : 0.05296
  Recon (MSE): 0.04305
  Gradient   : 0.02692
  Precip     : 0.10840
  Occupancy  : 0.08334



Epoch 050/50: 100%|██████████| 64/64 [00:05<00:00, 11.80it/s, Total=0.051, Recon=0.043, Precip=0.093]


Epoch 050/50 Summary:
  Total Loss : 0.05219
  Recon (MSE): 0.04243
  Gradient   : 0.02677
  Precip     : 0.10645
  Occupancy  : 0.08335

Обучение завершено!
Веса модели сохранены.


In [10]:
import os
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import xarray as xr
import numpy as np
from tqdm import tqdm

# ==========================================
# 1. АРХИТЕКТУРА VAE
# ==========================================
class ConvVAE(nn.Module):
    def __init__(self, in_channels=28, latent_channels=128):
        super(ConvVAE, self).__init__()
        
        self.enc1 = nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1)
        self.enc2 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)
        self.enc3 = nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1)
        
        self.fc_mu = nn.Conv2d(256, latent_channels, kernel_size=3, padding=1)
        self.fc_var = nn.Conv2d(256, latent_channels, kernel_size=3, padding=1)
        
        self.dec_input = nn.Conv2d(latent_channels, 256, kernel_size=3, padding=1)
        
        self.dec1 = nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1)
        self.dec2 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        self.dec3 = nn.ConvTranspose2d(64, in_channels, kernel_size=4, stride=2, padding=1)
        
    def encode(self, x):
        h = F.leaky_relu(self.enc1(x))
        h = F.leaky_relu(self.enc2(h))
        h = F.leaky_relu(self.enc3(h))
        return self.fc_mu(h), self.fc_var(h)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def quantize(self, z):
        # Квантование до целых чисел с помощью Straight-Through Estimator (STE)
        # Это позволяет градиентам проходить через недифференцируемую функцию округления при backpropagation
        z_rounded = torch.round(z)
        return z + (z_rounded - z).detach()

    def decode(self, z):
        h = F.relu(self.dec_input(z))
        h = F.relu(self.dec1(h))
        h = F.relu(self.dec2(h))
        return self.dec3(h)

    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        z_quantized = self.quantize(z)
        recon_x = self.decode(z_quantized)
        return recon_x, mu, log_var


# ==========================================
# 2. ФУНКЦИЯ ОЦЕНКИ
# ==========================================
def evaluate_on_train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Устройство: {device}")

    train_file = 'data/era5_highres_sample_2019.nc'
    if not os.path.exists(train_file):
        raise FileNotFoundError(f"Файл {train_file} не найден!")

    print("Загрузка обучающего датасета в память...")
    ds = xr.open_dataset(train_file)
    
    # 1. Подготовка широты для весов (Latitude-weighted)
    latitudes = ds.latitude.values
    cos_lat = np.clip(np.cos(np.deg2rad(latitudes)), a_min=0, a_max=None)
    cos_lat_tensor = torch.tensor(cos_lat, dtype=torch.float32, device=device).view(1, 1, -1, 1)

    surface_vars = [
        '2m_temperature', 'mean_sea_level_pressure', '10m_u_component_of_wind',
        '10m_v_component_of_wind', 'total_precipitation_6hr', 'sea_surface_temperature',
        'total_column_water_vapour', 'total_cloud_cover'
    ]
    atm_vars = ['temperature', 'u_component_of_wind', 'v_component_of_wind', 'geopotential', 'specific_humidity']

    # 2. Сборка тензоров (идентично процессу обучения)
    surf_arrays = [ds[v].values[:, np.newaxis, :, :] for v in surface_vars]
    atm_arrays = [ds[v].values for v in atm_vars]
    
    ds.close()
    del ds
    gc.collect()
    
    print("Склейка и нормализация данных...")
    data = np.concatenate(surf_arrays + atm_arrays, axis=1)
    del surf_arrays, atm_arrays
    gc.collect()

    data = np.nan_to_num(data, nan=0.0)
    
    # Считаем точные mean и std по всей выборке (это и есть sigma_train)
    data_mean = np.mean(data, axis=(0, 2, 3), keepdims=True)
    data_std = np.std(data, axis=(0, 2, 3), keepdims=True)
    data_std[data_std == 0] = 1.0 
    
    # Нормализуем данные
    data_normalized = (data - data_mean) / data_std
    data_normalized = data_normalized.astype(np.float32)

    H_orig, W_orig = data_normalized.shape[2], data_normalized.shape[3]
    pad_H = (8 - H_orig % 8) % 8
    pad_W = (8 - W_orig % 8) % 8
    data_padded = np.pad(data_normalized, ((0,0), (0,0), (0, pad_H), (0, pad_W)), mode='constant')

    # 3. Инициализация и загрузка модели
    model = ConvVAE(in_channels=28, latent_channels=128).to(device)
    try:
        model.load_state_dict(torch.load('models/era5_highres_vae_weights.pth', map_location=device))
        print("Веса модели успешно загружены.")
    except FileNotFoundError:
        print("ВНИМАНИЕ: Файл весов не найден. Модель будет выдавать случайные значения.")
    model.eval()

    # 4. Переменные для метрик
    num_channels = 28
    weighted_squared_errors = torch.zeros(num_channels, device=device)
    sum_of_weights = 0.0
    batch_size = 2
    num_samples = data_padded.shape[0]

    # Тензоры mean и std на GPU для денормализации
    mean_tensor = torch.tensor(data_mean, device=device)
    std_tensor = torch.tensor(data_std, device=device)

    print("\nНачало оценки батчей...")
    with torch.no_grad():
        for i in tqdm(range(0, num_samples, batch_size), desc="Оценка на Train"):
            x_batch = torch.tensor(data_padded[i : i+batch_size], device=device)
            
            # Оригинальные физические данные для этого батча (без паддинга)
            x_physical = torch.tensor(data[i : i+batch_size], device=device)

            with torch.amp.autocast('cuda'):
                recon_x, _, _ = model(x_batch)
            
            # Обрезаем паддинг и денормализуем
            recon_x_cropped = recon_x[:, :, :H_orig, :W_orig]
            recon_x_physical = recon_x_cropped * std_tensor + mean_tensor
            
            # Latitude-weighted MSE
            squared_error = (recon_x_physical - x_physical) ** 2
            weighted_se = squared_error * cos_lat_tensor
            
            weighted_squared_errors += weighted_se.sum(dim=(0, 2, 3))
            sum_of_weights += cos_lat_tensor.sum() * W_orig * x_batch.size(0)

    # 5. Итоговые расчеты
    print("\nРасчет NRMSE...")
    rmse_per_channel = torch.sqrt(weighted_squared_errors / sum_of_weights)
    
    # sigma_train — это вектор стандартных отклонений, который мы рассчитали выше (размерность: 28)
    sigma_f_train = std_tensor.squeeze() 
    
    nrmse_per_channel = rmse_per_channel / sigma_f_train
    
    nrmse_surface = nrmse_per_channel[:8]
    nrmse_pressure = nrmse_per_channel[8:]
    
    score_surface = nrmse_surface.mean().item()
    score_pressure = nrmse_pressure.mean().item()
    score_all = 0.5 * score_surface + 0.5 * score_pressure
    
    print("\n" + "="*45)
    print(" РЕЗУЛЬТАТЫ ОЦЕНКИ НА ОБУЧАЮЩИХ ДАННЫХ (2019)")
    print("="*45)
    print(f"Surface Score (S_surface) : {score_surface:.5f}")
    print(f"Pressure Score (S_pressure): {score_pressure:.5f}")
    print(f"Overall Score (S_all)     : {score_all:.5f}")
    print("-" * 45)
    
    print("Детализация NRMSE по наземным полям (Surface):")
    for i in range(8):
        print(f"  {surface_vars[i]:<25}: {nrmse_surface[i].item():.5f}")

if __name__ == "__main__":
    evaluate_on_train()

Устройство: cuda
Загрузка обучающего датасета в память...
Склейка и нормализация данных...
Веса модели успешно загружены.

Начало оценки батчей...


Оценка на Train: 100%|██████████| 64/64 [00:03<00:00, 17.67it/s]



Расчет NRMSE...

 РЕЗУЛЬТАТЫ ОЦЕНКИ НА ОБУЧАЮЩИХ ДАННЫХ (2019)
Surface Score (S_surface) : 0.23488
Pressure Score (S_pressure): 0.18064
Overall Score (S_all)     : 0.20776
---------------------------------------------
Детализация NRMSE по наземным полям (Surface):
  2m_temperature           : 0.15805
  mean_sea_level_pressure  : 0.08752
  10m_u_component_of_wind  : 0.19490
  10m_v_component_of_wind  : 0.21252
  total_precipitation_6hr  : 0.38944
  sea_surface_temperature  : 0.31077
  total_column_water_vapour: 0.17971
  total_cloud_cover        : 0.34616


In [11]:
import os
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import xarray as xr
import numpy as np
from tqdm import tqdm

# ==========================================
# 1. АРХИТЕКТУРА VAE
# ==========================================
class ConvVAE(nn.Module):
    def __init__(self, in_channels=28, latent_channels=128):
        super(ConvVAE, self).__init__()
        
        self.enc1 = nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1)
        self.enc2 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)
        self.enc3 = nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1)
        
        self.fc_mu = nn.Conv2d(256, latent_channels, kernel_size=3, padding=1)
        self.fc_var = nn.Conv2d(256, latent_channels, kernel_size=3, padding=1)
        
        self.dec_input = nn.Conv2d(latent_channels, 256, kernel_size=3, padding=1)
        
        self.dec1 = nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1)
        self.dec2 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        self.dec3 = nn.ConvTranspose2d(64, in_channels, kernel_size=4, stride=2, padding=1)
        
    def encode(self, x):
        h = F.leaky_relu(self.enc1(x))
        h = F.leaky_relu(self.enc2(h))
        h = F.leaky_relu(self.enc3(h))
        return self.fc_mu(h), self.fc_var(h)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def quantize(self, z):
        # Квантование до целых чисел с помощью Straight-Through Estimator (STE)
        # Это позволяет градиентам проходить через недифференцируемую функцию округления при backpropagation
        z_rounded = torch.round(z)
        return z + (z_rounded - z).detach()

    def decode(self, z):
        h = F.relu(self.dec_input(z))
        h = F.relu(self.dec1(h))
        h = F.relu(self.dec2(h))
        return self.dec3(h)

    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        z_quantized = self.quantize(z)
        recon_x = self.decode(z_quantized)
        return recon_x, mu, log_var
# ==========================================
# 2. ФУНКЦИЯ ОЦЕНКИ
# ==========================================
def evaluate_on_test():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Используемое устройство: {device}")

    test_file = 'data/era5_highres_test_2020.nc'
    if not os.path.exists(test_file):
        raise FileNotFoundError(f"Тестовый файл {test_file} не найден. Сначала скачайте его.")

    print("Загрузка тестового датасета в оперативную память...")
    ds = xr.open_dataset(test_file)
    
    # 1. Подготовка широты для весов (Latitude-weighted)
    latitudes = ds.latitude.values
    # Ограничиваем косинус снизу нулем для избежания отрицательных весов из-за погрешностей
    cos_lat = np.clip(np.cos(np.deg2rad(latitudes)), a_min=0, a_max=None)
    cos_lat_tensor = torch.tensor(cos_lat, dtype=torch.float32, device=device).view(1, 1, -1, 1)

    surface_vars = [
        '2m_temperature', 'mean_sea_level_pressure', '10m_u_component_of_wind',
        '10m_v_component_of_wind', 'total_precipitation_6hr', 'sea_surface_temperature',
        'total_column_water_vapour', 'total_cloud_cover'
    ]
    atm_vars = ['temperature', 'u_component_of_wind', 'v_component_of_wind', 'geopotential', 'specific_humidity']

    # 2. Сборка тензоров (28 каналов)
    surf_arrays = [ds[v].values[:, np.newaxis, :, :] for v in surface_vars]
    atm_arrays = [ds[v].values for v in atm_vars]
    
    ds.close()
    
    print("Склейка тензоров...")
    data = np.concatenate(surf_arrays + atm_arrays, axis=1)
    del surf_arrays, atm_arrays
    gc.collect()

    data = np.nan_to_num(data, nan=0.0)
    
    # Примечание: Строго говоря, для NRMSE нужна sigma_train. 
    # Так как скрипт должен быть независимым, мы аппроксимируем её стандартным отклонением 
    # текущей тестовой выборки, что для ERA5 даст практически идентичный результат.
    data_mean = np.mean(data, axis=(0, 2, 3), keepdims=True)
    data_std = np.std(data, axis=(0, 2, 3), keepdims=True)
    data_std[data_std == 0] = 1.0 
    
    # Z-нормализация
    data_normalized = (data - data_mean) / data_std
    data_normalized = data_normalized.astype(np.float32)

    # Паддинг до кратного 8
    H_orig, W_orig = data_normalized.shape[2], data_normalized.shape[3]
    pad_H = (8 - H_orig % 8) % 8
    pad_W = (8 - W_orig % 8) % 8
    data_padded = np.pad(data_normalized, ((0,0), (0,0), (0, pad_H), (0, pad_W)), mode='constant')

    # 3. Инициализация модели
    model = ConvVAE(in_channels=28, latent_channels=128).to(device)
    weights_path = 'models/era5_highres_vae_weights.pth'
    try:
        model.load_state_dict(torch.load(weights_path, map_location=device))
        print("Веса модели успешно загружены.")
    except FileNotFoundError:
        print(f"ВНИМАНИЕ: Файл {weights_path} не найден! Модель не обучена.")
    
    model.eval()

    # 4. Переменные для накопления метрик
    num_channels = 28
    weighted_squared_errors = torch.zeros(num_channels, device=device)
    sum_of_weights = 0.0
    
    batch_size = 2 # Поддерживаем размер 2 для RTX 5070
    num_samples = data_padded.shape[0]

    mean_tensor = torch.tensor(data_mean, device=device)
    std_tensor = torch.tensor(data_std, device=device)

    print("\nНачало прогона тестовой выборки...")
    with torch.no_grad():
        for i in tqdm(range(0, num_samples, batch_size), desc="Оценка батчей"):
            x_batch = torch.tensor(data_padded[i : i+batch_size], device=device)
            x_physical = torch.tensor(data[i : i+batch_size], device=device)

            # Прогон через модель со смешанной точностью
            with torch.amp.autocast('cuda'):
                recon_x, _, _ = model(x_batch)
            
            # Удаляем паддинг и возвращаем в физические единицы
            recon_x_cropped = recon_x[:, :, :H_orig, :W_orig]
            recon_x_physical = recon_x_cropped * std_tensor + mean_tensor
            
            # Вычисление взвешенного квадрата ошибки
            squared_error = (recon_x_physical - x_physical) ** 2
            weighted_se = squared_error * cos_lat_tensor
            
            # Накопление суммы числителя и знаменателя для RMSE
            weighted_squared_errors += weighted_se.sum(dim=(0, 2, 3))
            sum_of_weights += cos_lat_tensor.sum() * W_orig * x_batch.size(0)

    # 5. Итоговый расчет физических метрик
    print("\nПодведение итогов...")
    
    # RMSE с учетом широты
    rmse_per_channel = torch.sqrt(weighted_squared_errors / sum_of_weights)
    
    # NRMSE (нормализация на sigma)
    sigma_f_train = std_tensor.squeeze() 
    nrmse_per_channel = rmse_per_channel / sigma_f_train
    
    # Разделение скоров
    nrmse_surface = nrmse_per_channel[:8]
    nrmse_pressure = nrmse_per_channel[8:]
    
    score_surface = nrmse_surface.mean().item()
    score_pressure = nrmse_pressure.mean().item()
    score_all = 0.5 * score_surface + 0.5 * score_pressure
    
    print("\n" + "="*50)
    print(" РЕЗУЛЬТАТЫ ОЦЕНКИ НА ТЕСТОВОЙ ВЫБОРКЕ (2020)")
    print("="*50)
    print(f"Surface Score (S_surface) : {score_surface:.5f}")
    print(f"Pressure Score (S_pressure): {score_pressure:.5f}")
    print(f"Overall Score (S_all)     : {score_all:.5f}")
    print("-" * 50)
    
    print("Детализация NRMSE по наземным полям (Surface):")
    for i in range(8):
        print(f"  {surface_vars[i]:<25}: {nrmse_surface[i].item():.5f}")

if __name__ == "__main__":
    evaluate_on_test()

Используемое устройство: cuda
Загрузка тестового датасета в оперативную память...
Склейка тензоров...
Веса модели успешно загружены.

Начало прогона тестовой выборки...


Оценка батчей: 100%|██████████| 64/64 [00:04<00:00, 15.28it/s]



Подведение итогов...

 РЕЗУЛЬТАТЫ ОЦЕНКИ НА ТЕСТОВОЙ ВЫБОРКЕ (2020)
Surface Score (S_surface) : 0.23743
Pressure Score (S_pressure): 0.18476
Overall Score (S_all)     : 0.21109
--------------------------------------------------
Детализация NRMSE по наземным полям (Surface):
  2m_temperature           : 0.16578
  mean_sea_level_pressure  : 0.09250
  10m_u_component_of_wind  : 0.19816
  10m_v_component_of_wind  : 0.21963
  total_precipitation_6hr  : 0.39372
  sea_surface_temperature  : 0.31100
  total_column_water_vapour: 0.17175
  total_cloud_cover        : 0.34688
